In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import os

In [2]:
# Training and validation loss

def make_loss_plot(log_path, output_dir): 

    train_losses = []
    val_losses = []

    pattern = re.compile(r"Train Loss:\s*([0-9eE\.\-]+).*Val Loss:\s*([0-9eE\.\-]+)")

    with open(log_path, "r") as f:
        for line in f:
            m = pattern.search(line)
            if m:
                train_losses.append(float(m.group(1)))
                val_losses.append(float(m.group(2)))

    print("Extracted epochs:", len(train_losses))
    if len(train_losses) == 0:
        raise ValueError("No Train/Val loss found. Check log format / pattern.")

    epochs = list(range(1, len(train_losses) + 1))

    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(epochs, train_losses, label="Train Loss", linewidth=2)
    plt.plot(epochs, val_losses, label="Val Loss", linewidth=2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt_path = os.path.join(output_dir, "model_loss.png")
    plt.savefig(plt_path, dpi=300, bbox_inches="tight")
    plt.close()

In [8]:
# Deviation plot (True label vs predicted probablity)

def make_deviation_plot(preds_path, output_dir): 
    
    df = pd.read_csv(preds_path)

    # Sort so true=0 cases appear first (optional but cleaner)
    df_sorted = df.sort_values(["true_label", "case_id"]).reset_index(drop=True)
    df_sorted["case_str"] = df_sorted["case_id"].astype(str)

    # -----------------------------------------------------
    # 1. Deviation Plot (True vs Probabilities)
    # -----------------------------------------------------
    plt.figure(figsize=(5,7), dpi=150)

    for i, row in df_sorted.iterrows():
        # connecting line
        plt.plot(
            [row["true_label"], row["prob_class1"]],
            [i, i],
            color="gray",
            alpha=0.6,
            linewidth=1.5
        )

        # true label = large green circle
        plt.scatter(
            row["true_label"], i,
            color="green",
            s=200,
            marker="o",
            edgecolor="black",
            linewidth=0,
            zorder=3
        )

        # predicted probability = small black X
        plt.scatter(
            row["prob_class1"], i,
            color="black",
            s=120,
            marker="x",
            linewidths=2,
            zorder=4
        )

    plt.yticks(range(len(df_sorted)), df_sorted["case_str"])
    plt.axvline(0.5, color="#ADEBB3", linestyle="--", linewidth=2, label="Decision Threshold (0.5)")
    plt.xlabel("True Label (0 or 1)  <->  Predicted Probability (High-grade)")
    plt.ylabel("Case (sorted by True Label)")
    plt.title("Deviation Plot: True Label vs Predicted Probability")
    plt.grid(alpha=0.3, linestyle="--")
    plt.tight_layout()
    plt_path = os.path.join(output_dir, "deviation_plot.png")
    plt.savefig(plt_path, dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
output_dir = "runs/20260304_113837_MIL_split_3_final"
make_loss_plot("logs/851671_MIL_split_3_final.log", output_dir)
make_deviation_plot(os.path.join(output_dir, "predictions.csv"), output_dir)

Extracted epochs: 11
